In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                module_dir = self._local_data / module
                local_path = module_dir / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
                # Bundled data is present for this module but the file is absent
                # → it is genuinely missing. Raise immediately instead of falling
                # back to a Drive download that would hang offline (and pulls the
                # whole folder online just to discover the file is not there).
                if module_dir.is_dir():
                    raise FileNotFoundError(
                        f"File not found: {module}/{filename} "
                        f"(not in local data/{module}/)."
                    )
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp03/ex_2.py ──
"""
Shared infrastructure for MLFP03 Exercise 2 — Regularisation and
Cross-Validation.

Contains: data loading for the Singapore credit scoring dataset,
feature preparation, synthetic 1D bias-variance problem, alpha grids,
plotting helpers, and a shared OUTPUT_DIR for generated artefacts.

Technique-specific code (Ridge/Lasso model construction, nested CV
loops, learning curves) lives in the per-technique solution files —
this module holds only the helpers those files share.
"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from sklearn.base import clone
from sklearn.linear_model import LinearRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# CONSTANTS
# ════════════════════════════════════════════════════════════════════════

# Deterministic seed for every random operation in this exercise
SEED = 42

# Alpha sweep used by Ridge and the Ridge regularisation path.
ALPHAS: list[float] = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

# Lasso / ElasticNet need their OWN grid: sklearn's Lasso divides the
# squared-error term by 2n, so its α lives on a much smaller scale than
# Ridge's. With a standardised target, α ≥ ~0.5 already zeroes every
# coefficient; the grid below walks the full path from "almost OLS" to
# "intercept only".
LASSO_ALPHAS: list[float] = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0]

# Regression target for the credit demos. ``savings_balance`` is only
# partly predictable from the other columns (R² ≈ 0.25) and no single
# feature duplicates it — so regularisation has real work to do. (Contrast
# ``credit_utilization``: it is 0.97-correlated with
# ``avg_balance_utilization``, so every model would score the same and no
# regularisation effect would be visible.)
CREDIT_TARGET = "savings_balance"

# Columns that are never features in this exercise:
#   customer_id              — row identifier
#   default                  — the loan OUTCOME, observed after the fact
#   future_default_indicator — post-outcome copy of ``default`` (a planted
#                              leak; see Lesson 3.1)
CREDIT_NON_FEATURE_COLUMNS: tuple[str, ...] = (
    "customer_id",
    "default",
    "future_default_indicator",
)

# A SMALL training sample is deliberate: with ~33 correlated features and
# a few hundred rows, OLS visibly overfits (train R² well above test R²),
# which is exactly the regime where Ridge and Lasso earn their keep. The
# large test set keeps every test score stable to ~±0.01.
N_TRAIN_DEFAULT = 300
N_TEST_DEFAULT = 5000

# Output directory for HTML plots, summary CSVs, etc.
OUTPUT_DIR = Path("outputs") / "mlfp03_ex2_regularisation_cv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore credit scoring
# ════════════════════════════════════════════════════════════════════════


def load_credit_data(
    n_train: int = N_TRAIN_DEFAULT,
    n_test: int = N_TEST_DEFAULT,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, list[str]]:
    """Load + preprocess the MLFP02 Singapore credit scoring parquet.

    Returns:
        X_train, y_train, X_test, y_test, feature_names

    Target: ``savings_balance`` (S$), STANDARDISED with the training mean
    and standard deviation — so an MSE of 1.0 means "no better than
    predicting the mean" and MSE = 1 − R² (approximately) on the test set.

    ``n_train`` rows are used for fitting (small on purpose — see
    ``N_TRAIN_DEFAULT``) and ``n_test`` held-out rows for evaluation.
    ID, outcome and leak columns (``CREDIT_NON_FEATURE_COLUMNS``) are
    dropped before preprocessing.

    Uses ``kailash_ml.PreprocessingPipeline`` for normalisation +
    ordinal encoding + median imputation, fitted on the ``n_train``
    training rows only — the test rows are held out FIRST, so none of
    their statistics reach the scaler or the imputer. All regularised
    models REQUIRE normalised features (otherwise the penalty is unevenly
    distributed across the coefficient vector).
    """
    loader = MLFPDataLoader()
    credit = loader.load("mlfp02", "sg_credit_scoring.parquet").drop(
        CREDIT_NON_FEATURE_COLUMNS
    )
    credit = credit.sample(n=n_train + n_test, seed=SEED)

    # Split FIRST (exactly n_test held-out rows), then fit the preprocessing
    # on the training rows only.
    result = split_then_preprocess(
        credit,
        target=CREDIT_TARGET,
        test_size=n_test,
        seed=SEED,
        normalize=True,
        categorical_encoding="ordinal",
        imputation_strategy="median",
    )

    feature_cols = [c for c in result.train_data.columns if c != CREDIT_TARGET]
    X_train, y_train, col_info = to_sklearn_input(
        result.train_data,
        feature_columns=feature_cols,
        target_column=CREDIT_TARGET,
    )
    X_test, y_test, _ = to_sklearn_input(
        result.test_data,
        feature_columns=feature_cols,
        target_column=CREDIT_TARGET,
    )

    # Standardise the target with TRAINING statistics only (no test leakage).
    y_mean, y_std = float(y_train.mean()), float(y_train.std())
    y_train = (y_train - y_mean) / y_std
    y_test = (y_test - y_mean) / y_std

    return X_train, y_train, X_test, y_test, col_info["feature_columns"]


# Preprocessing for the credit-default CV demo. It is fitted INSIDE every
# fold (see ``cross_val_auc_split_first``), never once on the whole sample.
CREDIT_DEFAULT_PREPROCESSING: dict[str, Any] = {
    "normalize": True,
    "categorical_encoding": "ordinal",
    "imputation_strategy": "median",
}


def load_credit_default_sample(
    n: int = 600,
) -> tuple[pl.DataFrame, np.ndarray, list[str]]:
    """A small credit sample with the BINARY ``default`` outcome as target.

    Used to show why stratified k-fold matters for an imbalanced target
    (~13% defaults): with plain k-fold the default rate drifts from fold
    to fold. Returns (X_raw, y, feature_names). The ID / post-outcome leak
    columns are dropped, but ``X_raw`` is NOT preprocessed: in
    cross-validation every fold is its own train/test split, so imputation
    and scaling must be fitted inside each fold on that fold's training
    rows — fitting them once on all ``n`` rows would leak every test fold
    into its training fold. Score models with ``cross_val_auc_split_first``.
    """
    loader = MLFPDataLoader()
    credit = (
        loader.load("mlfp02", "sg_credit_scoring.parquet")
        .drop(["customer_id", "future_default_indicator"])
        .sample(n=n, seed=SEED)
    )
    y = credit["default"].to_numpy().astype(int)
    X_raw = credit.drop("default")
    return X_raw, y, X_raw.columns


def cross_val_auc_split_first(
    model: Any, X_raw: pl.DataFrame, y: np.ndarray, *, cv: Any
) -> np.ndarray:
    """ROC-AUC per fold with the preprocessing re-fitted inside every fold.

    Plays the role of ``cross_val_score(model, X, y, cv=cv,
    scoring="roc_auc")`` for RAW features: in each fold the
    ``PreprocessingPipeline`` is fitted on the training rows only and
    applied to the held-out rows, then a fresh clone of ``model`` is fitted
    and scored. Returns one AUC per fold.
    """
    frame = X_raw.with_columns(pl.Series("default", y))
    scores: list[float] = []
    for tr_idx, te_idx in cv.split(np.zeros(len(y)), y):
        prep = preprocess_train_test(
            frame[tr_idx.tolist()],
            frame[te_idx.tolist()],
            "default",
            seed=SEED,
            **CREDIT_DEFAULT_PREPROCESSING,
        )
        features = [c for c in prep.train_data.columns if c != "default"]
        X_tr, y_tr, _ = to_sklearn_input(
            prep.train_data, feature_columns=features, target_column="default"
        )
        X_te, y_te, _ = to_sklearn_input(
            prep.test_data, feature_columns=features, target_column="default"
        )
        fitted = clone(model).fit(X_tr, y_tr.astype(int))
        scores.append(float(roc_auc_score(y_te.astype(int), fitted.predict_proba(X_te)[:, 1])))
    return np.array(scores)


ICU_CV_FEATURES: list[str] = [
    "age",
    "gender",
    "height_cm",
    "weight_kg",
    "bmi",
    "insurance",
    "ethnicity",
    "diagnosis",
    "icu_type",
]


def load_icu_admissions_for_cv() -> dict[str, Any]:
    """ICU admissions with REAL time order and REAL repeated-patient groups.

    One row per admission (8,000 admissions of ~4,000 patients), sorted by
    ``admit_time`` so that TimeSeriesSplit's "earlier rows train, later
    rows test" really means "past predicts future". Features are only
    things known AT admission (patient demographics, diagnosis, ICU type);
    the target is ``los_days`` (length of stay).

    Returns a dict with X, y, groups (patient_id per row), admit_time
    (polars Series), feature_names.
    """
    loader = MLFPDataLoader()
    admissions = loader.load("mlfp02", "icu_admissions.parquet")
    patients = loader.load("mlfp02", "icu_patients.parquet")
    frame = (
        admissions.join(patients, on="patient_id", how="left")
        .with_columns(
            pl.col("admit_time").str.to_datetime("%Y-%m-%d %H:%M:%S")
        )
        .sort("admit_time")
    )
    encoded = frame.with_columns(
        [
            pl.col(c).rank("dense").cast(pl.Float64)
            for c in ICU_CV_FEATURES
            if frame.schema[c] == pl.String
        ]
    ).with_columns(
        [pl.col(c).cast(pl.Float64).fill_null(pl.col(c).median()) for c in ICU_CV_FEATURES]
    )
    return {
        "X": encoded.select(ICU_CV_FEATURES).to_numpy(),
        "y": encoded["los_days"].to_numpy(),
        "groups": encoded["patient_id"].to_numpy(),
        "admit_time": encoded["admit_time"],
        "feature_names": list(ICU_CV_FEATURES),
    }


# ════════════════════════════════════════════════════════════════════════
# SYNTHETIC 1D PROBLEM — for bias/variance and polynomial fits
# ════════════════════════════════════════════════════════════════════════


def sine_truth(x: np.ndarray) -> np.ndarray:
    """The noiseless generating function f(x) = sin(2πx)."""
    return np.sin(2 * np.pi * np.asarray(x).ravel())


def sample_sine_training_set(
    n: int,
    noise_sigma: float,
    rng: np.random.Generator,
) -> tuple[np.ndarray, np.ndarray]:
    """Draw ONE fresh training set (x ~ U[0,1], y = sin(2πx) + ε).

    The bias-variance decomposition is defined over many independent
    training sets drawn from the same process — this is how we draw them.
    """
    x = rng.uniform(0, 1, n)
    y = sine_truth(x) + rng.normal(0, noise_sigma, n)
    return x.reshape(-1, 1), y


def make_sine_dataset(
    n_train: int = 40,
    n_test: int = 1000,
    noise_sigma: float = 0.2,
    seed: int = SEED,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, float]:
    """Generate a 1D noisy-sine regression problem.

    Returns:
        x_train_2d (n_train, 1), y_train (n_train,),
        x_test_2d (n_test, 1),   y_test  (n_test,),
        noise_variance (float, σ² — the irreducible error floor)

    The true function is ``y = sin(2πx) + ε`` with ε ~ N(0, σ²).
    A small training set (40 points) makes overfitting visible; a large,
    independently drawn test set (1,000 points) makes the test MSE a
    stable estimate rather than the noise of a handful of points.
    The noise variance is returned so callers can use it in the
    bias-variance decomposition (σ² is the "irreducible noise" term).
    """
    rng = np.random.default_rng(seed)
    x_train, y_train = sample_sine_training_set(n_train, noise_sigma, rng)
    x_test, y_test = sample_sine_training_set(n_test, noise_sigma, rng)
    return x_train, y_train, x_test, y_test, noise_sigma**2


def make_poly_pipeline(degree: int) -> Pipeline:
    """Polynomial-features + scaler + linear-regression pipeline."""
    return Pipeline(
        [
            ("poly", PolynomialFeatures(degree, include_bias=False)),
            ("scaler", StandardScaler()),
            ("lr", LinearRegression()),
        ]
    )


# ════════════════════════════════════════════════════════════════════════
# REPORTING HELPERS
# ════════════════════════════════════════════════════════════════════════


def print_header(title: str) -> None:
    """Print a banner so each phase of a technique file is easy to spot."""
    print("\n" + "=" * 72)
    print(f"  {title}")
    print("=" * 72)


def format_results_table(rows: list[dict[str, Any]], cols: list[str]) -> str:
    """Render a small table of dicts as fixed-width text."""
    header = "  ".join(f"{c:>12}" for c in cols)
    sep = "-" * len(header)
    body = "\n".join(
        "  ".join(
            f"{row[c]:>12.4f}" if isinstance(row[c], float) else f"{row[c]:>12}"
            for c in cols
        )
        for row in rows
    )
    return f"{header}\n{sep}\n{body}"


def save_html_plot(fig: Any, name: str) -> Path:
    """Write a plotly figure into OUTPUT_DIR and return the path."""
    path = OUTPUT_DIR / name
    fig.write_html(str(path))
    return path




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 2.5: Learning Curves and Diagnostic Playbook
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Read a learning curve to decide "more data" vs "better model"
#   - Compare OLS, Ridge, and Lasso learning curves on one dataset
#   - Recognise the canonical shapes: large gap (high variance),
#     converged low (high bias), converged high (good fit)
#   - Use learning curves to justify (or reject) data-collection spend
#   - Tie the entire exercise together with a decision playbook
#
# PREREQUISITES:
#   - 01 through 04 in this exercise
#
# ESTIMATED TIME: ~35 minutes
#
# TASKS (5-phase R10):
#   1. Theory — learning-curve shapes and what they mean
#   2. Build — three models to compare (OLS, Ridge, Lasso)
#   3. Train — sklearn.learning_curve for each
#   4. Visualise — train-vs-validation curves on a real sample-size axis
#   5. Apply — a telco's churn-data purchase decision
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.model_selection import KFold, learning_curve



## THEORY — Reading a Learning Curve


In [ ]:
# A learning curve plots model performance (train and validation) as a
# function of the training-set size. The canonical readings:
#
#   1. LARGE GAP — train score well above validation score
#      Diagnosis: HIGH VARIANCE (overfitting). The model memorises its
#      training rows. Remedies: more data (if the validation curve is
#      still rising), stronger regularisation, or a simpler model.
#
#   2. CONVERGED LOW — train and validation meet, but at a poor score
#      Diagnosis: HIGH BIAS (underfitting). Adding rows WON'T help: the
#      curves are already together. Remedies: richer features or a more
#      flexible model class.
#
#   3. CONVERGED HIGH — train and validation meet at a good score
#      Diagnosis: good fit. More data gives marginal returns.
#
# The GAP between the curves reflects variance; the LEVEL at which they
# converge reflects bias (how good this model class can get here).



## TASK 2 — BUILD the comparison models


Read a learning curve from its LAST points (computed, not assumed).


In [ ]:
# A learning curve needs room to grow, so we draw a 4,000-row training
# pool (instead of the 300 rows used in files 02–04). The α values are
# the ones 5-fold CV chose on the 300-row sample in files 02 and 03.

print_header("Learning Curves — OLS vs Ridge vs Lasso")
# TODO: load_credit_data with a 4,000-row training pool and 1,000 test rows.
X_train, y_train, X_test, y_test, feature_names = ____
print(f"Training pool: {X_train.shape}")

models = {
    "OLS (unregularised)": LinearRegression(),
    "Ridge (α=100)": Ridge(alpha=100.0),
    "Lasso (α=0.1)": Lasso(alpha=0.1, max_iter=50_000),
}

TRAIN_SIZES = [50, 100, 200, 400, 800, 1600, 3200]
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)


def diagnose(train_curve: np.ndarray, val_curve: np.ndarray) -> str:
    # TODO: gap = last train score minus last validation score;
    # still_rising = validation improved by more than 0.01 over the last step.
    gap = ____
    still_rising = ____
    if gap > 0.05:
        return (
            "HIGH VARIANCE — large train/validation gap"
            + ("; validation still rising, more data will help" if still_rising else "")
        )
    # TODO: which condition on the final validation score means "converged LOW"?
    if ____:
        return "HIGH BIAS — curves have met at a low score; more data won't help"
    return "GOOD FIT — curves have met at a good score"



## TASK 3 — TRAIN each model across growing training-set sizes


In [ ]:
all_curves: dict[str, dict[str, np.ndarray]] = {}
for name, model in models.items():
    # TODO: sklearn learning_curve over TRAIN_SIZES with cv=cv, scoring="r2".
    # It returns (train_sizes, train_scores, validation_scores).
    train_sizes, tr_scores, te_scores = ____
    all_curves[name] = {
        "sizes": train_sizes,
        "train_mean": tr_scores.mean(axis=1),
        "test_mean": te_scores.mean(axis=1),
        "train_std": tr_scores.std(axis=1),
        "test_std": te_scores.std(axis=1),
    }

    print_header(name)
    print(f"{'N':>8} {'Train R²':>10} {'Val R²':>10} {'Gap':>10}")
    print("-" * 40)
    for n_, tr, te in zip(train_sizes, tr_scores.mean(axis=1), te_scores.mean(axis=1)):
        print(f"{n_:>8} {tr:>10.4f} {te:>10.4f} {(tr - te):>10.4f}")
    small_gap = all_curves[name]["train_mean"][0] - all_curves[name]["test_mean"][0]
    print(f"  Gap at N={train_sizes[0]}: {small_gap:.3f}")
    print(
        f"  Reading at N={train_sizes[-1]}: "
        f"{diagnose(all_curves[name]['train_mean'], all_curves[name]['test_mean'])}"
    )



### Checkpoint 1


In [ ]:
assert len(train_sizes) == len(
    TRAIN_SIZES
), "Should have one entry per training-set size"
assert all(
    "test_mean" in c for c in all_curves.values()
), "Every model should record a test-mean curve"
print("\n[ok] Checkpoint 1 passed — learning curves computed for all models")



## TASK 4 — VISUALISE the three learning curves


In [ ]:
# One figure, the REAL training-set sizes on a log x-axis, solid lines
# for validation and dashed for train. This plot is the single best
# diagnostic to show a budget committee when asking for more data.

fig = go.Figure()
for name, curves in all_curves.items():
    # TODO: validation curve — x is the REAL training-set sizes.
    fig.add_trace(
        go.Scatter(
            x=____,
            y=____,
            mode="lines+markers",
            name=f"{name} — validation",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=curves["sizes"],
            y=curves["train_mean"],
            mode="lines",
            line={"dash": "dash"},
            name=f"{name} — train",
        )
    )
fig.update_layout(
    title="Learning curves — credit savings regression",
    xaxis_title="Training set size (samples, log scale)",
    yaxis_title="R²",
    xaxis_type="log",
    yaxis_range=[-0.5, 1.0],
)
plot_path = save_html_plot(fig, "learning_curves_ols_ridge_lasso.html")
print(f"\nSaved: {plot_path}")



### Checkpoint 2


Reading the curves (computed from the tables above):
  - Small data (N={TRAIN_SIZES[0]}): OLS validation R² {ols['test_mean'][0]:+.3f} vs
    Ridge {ridge['test_mean'][0]:+.3f}. With few rows OLS overfits badly and
    regularisation is worth the most.
  - Large data (N={TRAIN_SIZES[-1]}): OLS {ols['test_mean'][-1]:+.3f}, Ridge
    {ridge['test_mean'][-1]:+.3f}. The models converge; regularisation matters
    less as data grows.
  - OLS at full size: {diagnose(ols['train_mean'], ols['test_mean'])}.


In [ ]:
assert plot_path.exists(), "Learning-curve plot should be saved"
print("\n[ok] Checkpoint 2 passed — learning-curve plot written")

ols = all_curves["OLS (unregularised)"]
ridge = all_curves["Ridge (α=100)"]
print(
)
if diagnose(ols["train_mean"], ols["test_mean"]).startswith("HIGH BIAS"):
    print(
        "  A linear model on these features has reached its ceiling — to do\n"
        "  better, change the features or the model class, not the row count."
    )



## TASK 5 — APPLY: a telco's churn-data purchase decision


Learning curve shape        | Decision
----------------------------|----------------------------------------
Large gap, still rising     | BUY data (or regularise harder) — price
(high variance)             | the extrapolated lift against the cost
----------------------------|----------------------------------------
Converged at a low score    | PASS on data; invest in features or a
(high bias)                 | more flexible model
----------------------------|----------------------------------------
Converged at a good score   | PASS — the model is done


In [ ]:
# SCENARIO (illustrative): a Singapore mobile operator is deciding
# whether to license 18 more months of historical call-detail records,
# growing its labelled churn training set from ~180K to ~420K
# subscribers, for a one-off cost of ~S$1.4M.
#
# WHY LEARNING CURVES ARE THE RIGHT TOOL:
#   - The CTO needs a defensible answer to "will more data reduce
#     churn?" BEFORE the cheque is written.
#   - If the validation curve has CONVERGED with the train curve (as our
#     credit curves did), extra rows will not move the needle — spend
#     the money on BETTER features or a more flexible model instead.
#   - If a LARGE GAP remains and the validation curve is still RISING,
#     extrapolate the curve to the new size, convert the expected lift
#     into retained revenue with the retention team, and compare that
#     with the licence cost.

print_header("Telco Churn — learning-curve data decision")
print(
)



## REFLECTION — The Full Exercise in One Frame


======================================================================
  WHAT YOU'VE MASTERED (ENTIRE EXERCISE)
======================================================================

  01. Bias-Variance     — why "more complex" is not always "better"
  02. Ridge (L2)        — shrinkage, Gaussian prior, stability
  03. Lasso + ElasticNet — sparsity, L1 diamond, feature selection
  04. Cross-validation  — nested / stratified / time-series / group
  05. Learning curves   — diagnose data hunger vs model weakness

  DECISION PLAYBOOK:
    Step 1. Start with a learning curve on a regularised baseline.
    Step 2. Large train/validation gap and validation still rising →
            more data and/or stronger regularisation (HIGH VARIANCE).
    Step 3. Curves converged close together at a poor score → a richer
            model or better features, NOT more data (HIGH BIAS).
    Step 4. If the data has temporal or group structure, DO NOT use
            shuffled k-fold — use TimeSeriesSplit or GroupKFold.
    Step 5. Choose hyperparameters with (nested) CV, never on the test set.

  KEY INSIGHT: Regularisation is how you encode your prior belief about
  the world. Cross-validation is how you audit that belief against
  reality. Learning curves tell you whether reality will change if you
  throw more data at it.

  NEXT: Exercise 3 — full supervised model zoo (SVM, KNN, Naive Bayes,
  Trees, Random Forests) on Singapore e-commerce churn data.


In [ ]:
print(
)

